# Coding practice: zero-shot classification with CLIP

CLIP was trained on image-caption pairs until a picture and a description of it land close together (lecture 5.2).
That gives a classifier that needs no training at all: write a short description of each class, and give each
image the class whose description its vector is closest to. You will write that step, run it on 20 small images,
and then run the same code on two models that were trained apart.

The three models download the first time they run, about 1.4 GB in all. Everything runs on a CPU in seconds.

> Save your own copy first: File → Save a copy in Drive.

In [ ]:
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModel, AutoTokenizer, CLIPModel, CLIPProcessor


def features(output):
    """transformers 4 returns CLIP's vectors directly; transformers 5 returns an output that holds them."""
    return output if torch.is_tensor(output) else output.pooler_output

In [ ]:
#@title The 20 images: the first two CIFAR-10 test images of each class
import base64
import io

import numpy as np

_DATA = (
    "..."
)
with np.load(io.BytesIO(base64.b64decode(_DATA)), allow_pickle=False) as _f:
    IMAGES, LABELS = _f["images"], _f["labels"].tolist()
CLASSES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
PICTURES = [Image.fromarray(image) for image in IMAGES]
print(len(PICTURES), "images,", IMAGES.shape[1], "pixels square")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 10, figsize=(12, 2.6))
for ax, image in zip(axes.flat, IMAGES):
    ax.imshow(image)
    ax.axis("off")
plt.show()
print("Top row, left to right:", ", ".join(CLASSES[y] for y in LABELS[:10]))
print("Bottom row:            ", ", ".join(CLASSES[y] for y in LABELS[10:]))

## 1. Scale every vector to length 1

`normalize` scales every row of a matrix to length 1. After that, the dot product of two rows is their cosine
similarity, the alignment function from lecture 5.2.

<details><summary>Hint</summary>

A row's length is the square root of the sum of its squared entries. Divide each row by its own length;
`torch.linalg.norm(x, dim=1, keepdim=True)` gives every row's length as a column.

</details>

In [ ]:
def normalize(x):
    """x: a matrix with one vector per row. Return it with every row scaled to length 1."""
    # TODO: divide each row by its length.
    raise NotImplementedError("Scale the rows.")

In [ ]:
toy = torch.tensor([[3.0, 4.0], [0.0, 2.0]])
assert torch.allclose(normalize(toy), torch.tensor([[0.6, 0.8], [0.0, 1.0]])), normalize(toy)
assert torch.allclose(torch.linalg.norm(normalize(torch.randn(5, 7)), dim=1), torch.ones(5))
print("normalize works.")

## 2. Give each image its closest description

`zero_shot` takes one vector per image and one per class description, and returns, for each image, the index of the
class whose description has the highest cosine similarity with it.

<details><summary>Hint</summary>

Normalize both matrices. One matrix product then gives the cosine similarity of every image with every class, one
row per image: the similarity matrix from lecture 5.2. Each image's answer is the position of the largest entry in
its row, which `argmax(dim=1)` returns for every row at once.

</details>

In [ ]:
def zero_shot(image_vectors, text_vectors):
    """image_vectors: one row per image. text_vectors: one row per class. Return each image's class index."""
    # TODO: score every image against every class by cosine similarity, then pick each image's best class.
    raise NotImplementedError("Score and pick.")

In [ ]:
images = torch.tensor([[0.9, 0.1], [0.2, 0.8]])
assert zero_shot(images, torch.tensor([[1.0, 0.0], [0.0, 1.0]])).tolist() == [0, 1]
# A class whose vector is long must not win on its length alone.
assert zero_shot(images, torch.tensor([[10.0, 0.0], [0.0, 1.0]])).tolist() == [0, 1], "Compare directions, not lengths."
print("zero_shot works.")

## 3. Classify with CLIP

Each class becomes the description `a photo of a <class>.`, and CLIP turns the 20 images and the 10 descriptions
into vectors in one space. The images are 32 pixels square, far smaller than the photos CLIP learned from.

Predict: how many of the 20 will CLIP label correctly? Write your guess in the next cell before you run anything
after it.

In [ ]:
MY_GUESS_CLIP = None   # how many of the 20 CLIP labels correctly

In [ ]:
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").eval()
clip_inputs = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
PROMPTS = [f"a photo of a {name}." for name in CLASSES]
with torch.no_grad():
    CLIP_IMAGES = features(clip.get_image_features(**clip_inputs(images=PICTURES, return_tensors="pt")))
    CLIP_TEXTS = features(clip.get_text_features(**clip_inputs(text=PROMPTS, return_tensors="pt", padding=True)))
print("CLIP vectors:", tuple(CLIP_IMAGES.shape), "for the images,", tuple(CLIP_TEXTS.shape), "for the descriptions")

predicted = zero_shot(CLIP_IMAGES, CLIP_TEXTS)
correct = int((predicted == torch.tensor(LABELS)).sum())
print(f"CLIP labels {correct} of 20 correctly. You guessed {MY_GUESS_CLIP}.")
for i, (p, y) in enumerate(zip(predicted.tolist(), LABELS)):
    if p != y:
        print(f"  image {i}: a {CLASSES[y]}, labelled {CLASSES[p]}")

## 4. Two models that never met

DINOv2 learned from images alone, and MPNet from text alone: neither ever saw an image beside its caption. Both
produce 768 numbers per input, so your `zero_shot` runs on them unchanged, with DINOv2 vectors for the 20 images and
MPNet vectors for the same 10 descriptions.

Predict: with the same images, the same descriptions and the same code, how many of the 20 will these two models
label correctly? With 10 classes, chance is about 2.

In [ ]:
MY_GUESS_APART = None   # how many of the 20 DINOv2 with MPNet labels correctly

In [ ]:
dino = AutoModel.from_pretrained("facebook/dinov2-base").eval()
dino_inputs = AutoImageProcessor.from_pretrained("facebook/dinov2-base")
mpnet = AutoModel.from_pretrained("sentence-transformers/all-mpnet-base-v2").eval()
mpnet_inputs = AutoTokenizer.from_pretrained("sentence-transformers/all-mpnet-base-v2")
with torch.no_grad():
    # DINOv2's image vector: the mean of its patch vectors. MPNet's sentence vector: the mean over its tokens.
    DINO_IMAGES = dino(**dino_inputs(images=PICTURES, return_tensors="pt")).last_hidden_state[:, 1:].mean(1)
    tokens = mpnet_inputs(PROMPTS, return_tensors="pt", padding=True)
    mask = tokens["attention_mask"].unsqueeze(-1).float()
    MPNET_TEXTS = (mpnet(**tokens).last_hidden_state * mask).sum(1) / mask.sum(1)
print("DINOv2 vectors:", tuple(DINO_IMAGES.shape), " MPNet vectors:", tuple(MPNET_TEXTS.shape))

predicted = zero_shot(DINO_IMAGES, MPNET_TEXTS)
correct = int((predicted == torch.tensor(LABELS)).sum())
print(f"DINOv2 with MPNet labels {correct} of 20 correctly. You guessed {MY_GUESS_APART}. Chance is about 2.")

## Interpret what you found

1. CLIP missed at most an image or two. Look at any it missed: is the mistake one a person squinting at a 32-pixel
   image might make?
2. Your `zero_shot` ran unchanged on both pairs of models. What does a high cosine similarity between a CLIP image
   vector and a CLIP text vector tell you that the same number between a DINOv2 vector and an MPNet vector does not?
3. DINOv2 is a strong image model. Within its own space, is each image closest to the other image of its class? Try
   it: `normalize(DINO_IMAGES) @ normalize(DINO_IMAGES).T` holds every pairwise similarity.

Lab 5 asks whether two spaces like DINOv2's and MPNet's can be lined up without a single image-caption pair.

## Optional: your own image and your own classes

Upload a photo and describe a few classes of your own. Nothing here is graded.

In [ ]:
MY_DESCRIPTIONS = ["a photo of a dog.", "a photo of a bicycle.", "a photo of a coffee cup."]
try:
    from google.colab import files
except ImportError:
    print("Uploading works in Colab.")
else:
    uploaded = files.upload()
    if uploaded:
        photo = Image.open(io.BytesIO(next(iter(uploaded.values())))).convert("RGB")
        with torch.no_grad():
            mine = features(clip.get_image_features(**clip_inputs(images=[photo], return_tensors="pt")))
            texts = features(clip.get_text_features(**clip_inputs(text=MY_DESCRIPTIONS, return_tensors="pt", padding=True)))
        print("CLIP picks:", MY_DESCRIPTIONS[int(zero_shot(mine, texts)[0])])